# The DS inner loop with mbt

This notebook is the data scientist's lab bench for the wide batch-monthly churn cadence.
The philosophy: the notebook is where you EXPLORE and ANALYZE; the model itself always lives in reviewed YAML ("the model config IS the model").
Nothing trained from ad-hoc notebook calls can reach the registry - training goes through `mbt build`, which pins data snapshots, seeds, and gates, so a teammate or the scheduler reproduces your result without your notebook.

Run this inside the showcase's JupyterLab (`make up`, then http://localhost:8899, open `project/notebooks/`).
Background reading: `docs/tutorial.md` (the two-persona story), `docs/naming-conventions.md` (what `inference_date`, `as_of_date`, and `loaded_at_time` mean), and `DESIGN.md` in the showcase root.

In [ ]:
import json
import os
import subprocess
from pathlib import Path

import pandas as pd

# The kernel starts in notebooks/; every mbt command runs from the project root.
if not Path("mbt_project.yml").exists() and Path("/workspace/project/mbt_project.yml").exists():
    os.chdir("/workspace/project")
assert Path("mbt_project.yml").exists(), "run this notebook from the showcase JupyterLab"

# The showcase pins its anchor to the seeded data range. In production the
# scheduler passes the Airflow logical date here (execution_date = target_date
# = inference_date, docs/naming-conventions.md).
ANCHOR = "2026-06-30T00:00:00Z"
print("project root:", Path.cwd())

## 1. Explore the raw lake tables

The gold tables live in SeaweedFS (`s3://mbt-lake`, browse them at http://localhost:8388/buckets/).
The same parquet is staged on the shared mount at `/workspace/seed/`, which is the zero-dependency way to poke at it from a notebook.
Every wide table joins on ONE key, `inference_date`; the population spine also carries the entity crosswalk (`customer_id` to `safe_id`) and the informational `as_of_date` / `loaded_at_time` columns.

In [ ]:
population = pd.read_parquet("/workspace/seed/monthly_population")
labels = pd.read_parquet("/workspace/seed/monthly_labels")
print(f"population: {population.shape}, labels: {labels.shape}")
first, last = population["inference_date"].min(), population["inference_date"].max()
print(f"cohorts: {first:%Y-%m-%d} .. {last:%Y-%m-%d}")
print(f"churn rate over matured cohorts: {labels['is_churn'].mean():.1%}")
population.head()

## 2. The model is YAML

You do not define the training set in this notebook - you review the declaration.
The dataset spec below is everything: the population spine, per-table join keys, the matured-label contract, panel sampling, and the exact train/test cohort boundaries.
Editing any of it flips the model's config hash, so CI retrains exactly what changed.

In [ ]:
print(Path("datasets/wide_churn_training.yml").read_text())

## 3. The fast inner loop: build the probe on the dev target

`mbt build` compiles a pinned manifest, materializes the joined panel with Spark over the s3a lake, trains, evaluates gates, and writes machine-readable results.
Exit codes mean something: 0 trained and passed, 2 means a gate or check said no (feedback, not breakage), 1 is a real error.

In [ ]:
!mbt build --target dev --select churn_wide_probe --anchor 2026-06-30T00:00:00Z

## 4. Analyze what the run produced

Everything mbt writes is a file the notebook can read: metrics and gates in `target/run_results.json`, and the EXACT frame the model saw under `target/datasets/`.
This is where notebook strengths matter - slice the panel, inspect importances, question the gates.

In [ ]:
results = json.loads(Path("target/run_results.json").read_text())
probe = next(r for r in results["results"] if r["unique_id"].endswith(".churn_wide_probe"))
print("metrics:", {k: round(v, 4) for k, v in probe["metrics"].items()})
print("gates:", probe["gates"])
importance = pd.Series(probe["feature_importance"]).sort_values(ascending=False)
importance.head(10)

In [ ]:
newest = max(
    Path("target/datasets/wide_churn_training").glob("*/train.parquet"),
    key=lambda p: p.stat().st_mtime,
)
train = pd.read_parquet(newest)
print(f"train panel: {train.shape[0]} rows x {train.shape[1]} columns")
train.groupby("inference_date")[["login_days_30d", "txn_cnt_30d", "is_churn"]].mean().round(3)

## 5. Feature selection is a committed diff

`scripts/select_features.py` runs the ds-helper funnel over the materialized train split (drop high-missing, drop single-value, drop correlated pairs, then a seeded LightGBM randomized search keeping importance > 0) and rewrites the include list in `models/churn_wide_automl.yml`.
It honors the model's `exclude:` list - the DS ignored-columns contract - and it is deterministic: on the committed data it reproduces the committed list byte for byte, so this cell leaves no diff.

In [ ]:
!python scripts/select_features.py

In [ ]:
report = json.loads(Path("target/feature_selection_report.json").read_text())
stages = report["stages"]
print("candidates:", report["n_candidate_features"], "| excluded by contract:", report["excluded"])
print("high-missing dropped:", len(stages["high_missing"]["dropped"]))
print("single-value dropped:", len(stages["single_unique"]["dropped"]))
print("correlated dropped:", stages["correlated"]["dropped"])
print("zero-importance dropped:", len(stages["lgbm"]["zero_importance_dropped"]))
print("cv roc_auc:", round(stages["lgbm"]["best_cv_roc_auc"], 4))
pd.DataFrame(report["selected"])

## 6. Experiment on a sample without touching the contract

For quick what-ifs, hash sampling gives you a coherent slice: `sample_fraction: 0.25` keeps every snapshot of a quarter of the customers, and smaller fractions are subsets of larger ones.
Point the funnel at a SCRATCH copy of the model file so the committed contract stays clean while you compare selections.
(If you later want to regenerate the committed list, rebuild at full fraction first - the funnel always reads the newest complete materialization.)

In [ ]:
import shutil

scratch = Path("/tmp/sampled_experiment.yml")
shutil.copy("models/churn_wide_automl.yml", scratch)
shutil.copy("models/wide_hooks.py", "/tmp/wide_hooks.py")  # imported next to the model file

subprocess.run(
    [
        "mbt",
        "build",
        "--target",
        "dev",
        "--select",
        "churn_wide_probe",
        "--anchor",
        ANCHOR,
        "--vars",
        "sample_fraction: 0.25",
    ],
    check=True,
)
sampled_train = max(
    Path("target/datasets/wide_churn_training").glob("*/train.parquet"),
    key=lambda p: p.stat().st_mtime,
)
subprocess.run(
    [
        "python",
        "scripts/select_features.py",
        "--train-parquet",
        str(sampled_train),
        "--model-file",
        str(scratch),
        "--report",
        "/tmp/sampled_report.json",
    ],
    check=True,
)

committed = Path("models/churn_wide_automl.yml").read_text()
print("committed contract untouched:", "BEGIN selected-features" in committed)
print("sampled selection differs:", scratch.read_text() != committed)

## 7. Ship it: the notebook ends where the PR begins

When the spec and the selection look right, everything you decided is already sitting in files a reviewer can read: the dataset/model YAML, the rewritten include list, `hooks.py`, and the selection report.
Commit them on a branch and open a PR (in the showcase: clone `http://localhost:3305/mbt-showcase/churn` after `make ci`).
From there the platform takes over, and none of it needs this notebook:

- the PR check state-diffs against the published baseline and retrains ONLY what you changed, posting metrics vs the production champion as a comment;
- after merge, `make wide` (or the CI flow) trains sparkling AutoML on the cluster, the Evidently train gate checks the selected features for stability and BLOCKS promotion on a breach (exit 2), and gate-verified promotion flips the registry alias - zero redeploy;
- every 1st of the month the `mbt_score_wide` DAG scores the newest cohort with the run-time champion, re-checks stability against the exported baseline, and `mbt monitor` evaluates realized metrics once the labels mature.

Your experiment became infrastructure the moment you committed the YAML.